# 🚗 Big Data Analytics — Serviço de Transporte por Aplicativo
## ETL & Análise Exploratória de Dados (EDA)
**Dataset:** 150.000 registros de corridas | **Período:** 2024

---
### Arquitetura do Projeto
```
[ Fonte: XLSX Bruto ]
       ↓
[ Camada 1: Python — Extração, Limpeza, Transformação ]
       ↓
[ Camada 2: SQL — Modelagem Relacional & Views ]
       ↓
[ Camada 3: Power BI — Dashboard Executivo ]
```

## 1. Importação de Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import sqlite3
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style='whitegrid', palette='muted')

print('Bibliotecas carregadas com sucesso!')

## 2. Extração (E) — Carregamento do Dataset Bruto

In [ ]:
df_raw = pd.read_excel('Pasta1.xlsx', sheet_name='g7')

print(f'📦 Registros carregados: {len(df_raw):,}')
print(f'📊 Colunas: {df_raw.shape[1]}')
print()
df_raw.head()

In [ ]:
# Diagnóstico de nulos
nulls = df_raw.isnull().sum()
null_pct = (nulls / len(df_raw) * 100).round(2)
diag = pd.DataFrame({'Nulos': nulls, 'Percentual (%)': null_pct})
diag[diag['Nulos'] > 0].sort_values('Percentual (%)', ascending=False)

In [ ]:
print('📌 Distribuição dos Status:')
print(df_raw['Booking Status'].value_counts())
print(f"\n Taxa de cancelamento total: {((df_raw['Booking Status'].isin(['Cancelled by Driver','Cancelled by Customer'])).sum() / len(df_raw) * 100):.1f}%")

## 3. Transformação (T) — Limpeza e Enriquecimento

In [ ]:
df = df_raw.copy()

# 3.1 — Data e Hora
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df['Time'] = pd.to_datetime(df['Time'], format='%H:%M:%S', errors='coerce').dt.time

df['order_hour']    = pd.to_datetime(df['Time'].astype(str), format='%H:%M:%S', errors='coerce').dt.hour
df['order_month']   = df['Date'].dt.month
df['order_weekday'] = df['Date'].dt.day_name()
df['order_quarter'] = df['Date'].dt.quarter

print(' 3.1 — Data/Hora convertidas e colunas temporais criadas.')

In [ ]:
# 3.2 — Avg VTAT e Avg CTAT: imputação por mediana do Vehicle Type
for col in ['Avg VTAT', 'Avg CTAT']:
    medians = df.groupby('Vehicle Type')[col].transform('median')
    df[col] = df[col].fillna(medians).fillna(df[col].median())

print(' 3.2 — Avg VTAT e Avg CTAT tratados.')
print(f"   VTAT nulos restantes: {df['Avg VTAT'].isnull().sum()}")
print(f"   CTAT nulos restantes: {df['Avg CTAT'].isnull().sum()}")

In [ ]:
# 3.3 — Valores numéricos
df['Booking Value'] = pd.to_numeric(df['Booking Value'], errors='coerce').fillna(0)
df['Ride Distance'] = pd.to_numeric(df['Ride Distance'], errors='coerce').fillna(0)

print(' 3.3 — Booking Value e Ride Distance convertidos.')
print(df['Booking Value'].describe())

In [ ]:
# 3.4 — Motivos de cancelamento
df['Driver Cancellation Reason']        = df['Driver Cancellation Reason'].fillna('N/A')
df['Reason for cancelling by Customer'] = df['Reason for cancelling by Customer'].fillna('N/A')
df['Incomplete Rides Reason']           = df['Incomplete Rides Reason'].fillna('N/A')
df['Payment Method']                    = df['Payment Method'].fillna('Não Informado')

def unify_cancel_reason(row):
    if row['Booking Status'] == 'Cancelled by Driver':
        return '[Motorista] ' + str(row['Driver Cancellation Reason'])
    elif row['Booking Status'] == 'Cancelled by Customer':
        return '[Cliente] ' + str(row['Reason for cancelling by Customer'])
    elif row['Booking Status'] == 'No Driver Found':
        return '[Sistema] Nenhum Motorista Disponível'
    elif row['Booking Status'] == 'Incomplete':
        return '[Incompleta] ' + str(row['Incomplete Rides Reason'])
    return 'N/A - Concluída'

df['cancel_reason_unified'] = df.apply(unify_cancel_reason, axis=1)

print(' 3.4 — Motivos de cancelamento unificados.')

In [ ]:
# 3.5 — Flags binárias (necessárias para as views SQL)
df['is_completed']          = (df['Booking Status'] == 'Completed').astype(int)
df['cancelled_by_driver']   = (df['Booking Status'] == 'Cancelled by Driver').astype(int)
df['cancelled_by_customer'] = (df['Booking Status'] == 'Cancelled by Customer').astype(int)
df['no_driver_found']       = (df['Booking Status'] == 'No Driver Found').astype(int)

# Categoria de TAT
df['vtat_category'] = pd.cut(
    df['Avg VTAT'],
    bins=[0, 3, 6, 10, float('inf')],
    labels=['Rapido', 'Normal', 'Lento', 'Critico']
).astype(str)

print(' 3.5 — Flags binárias criadas.')
print(f"   is_completed          : {df['is_completed'].sum():,}")
print(f"   cancelled_by_driver   : {df['cancelled_by_driver'].sum():,}")
print(f"   cancelled_by_customer : {df['cancelled_by_customer'].sum():,}")
print(f"   no_driver_found       : {df['no_driver_found'].sum():,}")

In [ ]:
# 3.6 — Renomear TODAS as colunas para snake_case
df.columns = [col.strip().lower().replace(' ', '_') for col in df.columns]

print(' 3.6 — Colunas renomeadas para snake_case.')
print('\nColunas finais:')
for c in df.columns:
    print(f'  - {c}')

## 4. Análise Exploratória (EDA)

In [ ]:
# 4.1 — Status das corridas
status_counts = df['booking_status'].value_counts()
colors = ['#2ecc71','#e74c3c','#e67e22','#3498db','#95a5a6']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].pie(status_counts, labels=status_counts.index, colors=colors,
            autopct='%1.1f%%', startangle=140,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[0].set_title('Distribuição dos Status das Corridas', fontsize=14, fontweight='bold')

status_counts.plot(kind='barh', ax=axes[1], color=colors, edgecolor='white')
axes[1].set_title('Quantidade por Status', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Quantidade de Corridas')
for i, v in enumerate(status_counts):
    axes[1].text(v + 300, i, f'{v:,}', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('chart_status.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 4.2 — Receita por tipo de veículo
revenue_vehicle = df.groupby('vehicle_type')['booking_value'].sum().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
revenue_vehicle.plot(kind='barh', ax=ax, color='#3498db', edgecolor='white')
ax.set_title('Faturamento Total por Tipo de Veículo', fontsize=14, fontweight='bold')
ax.set_xlabel('Receita Total (R$)')
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'R$ {x/1e6:.1f}M'))
for i, v in enumerate(revenue_vehicle):
    ax.text(v + 100000, i, f'R$ {v/1e6:.2f}M', va='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig('chart_receita_veiculo.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 4.3 — Horários de pico (No Driver Found)
total_by_hour    = df.groupby('order_hour').size()
nodriver_by_hour = df.groupby('order_hour')['no_driver_found'].sum()
nodriver_rate    = (nodriver_by_hour / total_by_hour * 100).fillna(0)

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(nodriver_rate.index, nodriver_rate.values, color='#e74c3c', alpha=0.8, edgecolor='white')
ax.set_title('Taxa "Nenhum Motorista Encontrado" por Hora do Dia (%)', fontsize=14, fontweight='bold')
ax.set_xlabel('Hora do Dia')
ax.set_ylabel('Taxa (%)')
ax.set_xticks(range(0, 24))
ax.axhline(nodriver_rate.mean(), color='navy', linestyle='--', label=f'Média: {nodriver_rate.mean():.1f}%')
ax.legend()

plt.tight_layout()
plt.savefig('chart_nodriver_hora.png', dpi=150, bbox_inches='tight')
plt.show()

print('🔴 Top 5 horas críticas:')
print(nodriver_rate.sort_values(ascending=False).head())

In [ ]:
# 4.4 — Motivos de cancelamento pelo motorista
driver_cancel = df[df['booking_status'] == 'Cancelled by Driver']
reason_counts = driver_cancel['driver_cancellation_reason'].value_counts()

fig, ax = plt.subplots(figsize=(10, 5))
reason_counts.plot(kind='bar', ax=ax,
                   color=['#e74c3c','#e67e22','#f39c12','#d35400'], edgecolor='white')
ax.set_title('Motivos de Cancelamento pelo Motorista', fontsize=14, fontweight='bold')
ax.set_ylabel('Quantidade')
ax.set_xticklabels(reason_counts.index, rotation=15, ha='right')
for i, v in enumerate(reason_counts):
    ax.text(i, v + 50, f'{v:,}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('chart_motivos_cancelamento.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 4.5 — VTAT x Taxa de não conclusão
vtat_cancel = df.groupby('vtat_category').agg(
    total=('booking_id', 'count'),
    cancelamentos=('is_completed', lambda x: (1 - x).sum())
).reset_index()
vtat_cancel['taxa_cancel'] = vtat_cancel['cancelamentos'] / vtat_cancel['total'] * 100

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(vtat_cancel['vtat_category'].astype(str), vtat_cancel['taxa_cancel'],
              color=['#2ecc71','#f39c12','#e67e22','#e74c3c'], edgecolor='white')
ax.set_title('Taxa de Não Conclusão por Categoria VTAT', fontsize=13, fontweight='bold')
ax.set_ylabel('Taxa (%)')
ax.set_xlabel('Categoria VTAT')
for bar, val in zip(bars, vtat_cancel['taxa_cancel']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('chart_vtat_cancelamento.png', dpi=150, bbox_inches='tight')
plt.show()
print('📌 Quanto maior o VTAT, maior a taxa de não conclusão.')

In [ ]:
# 4.6 — Top 10 locais por receita
top_locations = df.groupby('pickup_location')['booking_value'].sum().sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(12, 6))
top_locations.sort_values().plot(kind='barh', ax=ax, color='#2980b9', edgecolor='white')
ax.set_title('Top 10 Locais de Origem por Receita Total', fontsize=14, fontweight='bold')
ax.set_xlabel('Receita Total (R$)')
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'R$ {x/1e3:.0f}K'))

plt.tight_layout()
plt.savefig('chart_top_locations.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Carga (L) — Exportação para SQLite e CSV

In [ ]:
# 5.1 — Preparar DataFrame para exportação
df_export = df.copy()
df_export['date'] = df_export['date'].dt.strftime('%Y-%m-%d')
df_export['time'] = df_export['time'].astype(str)
df_export['vtat_category'] = df_export['vtat_category'].astype(str)

print(f' DataFrame pronto: {df_export.shape[0]:,} linhas x {df_export.shape[1]} colunas')

In [ ]:
# 5.2 — Exportar CSV para Power BI
df_export.to_csv('transporte_limpo.csv', index=False, encoding='utf-8-sig')
print(f' CSV exportado: transporte_limpo.csv ({len(df_export):,} linhas)')

In [ ]:
# 5.3 — Salvar no banco SQLite (transporte_v2.db)
conn = sqlite3.connect('transporte_v2.db')
df_export.to_sql('fato_corridas', conn, if_exists='replace', index=False)

total = pd.read_sql('SELECT COUNT(*) as n FROM fato_corridas', conn).iloc[0,0]
colunas = pd.read_sql('PRAGMA table_info(fato_corridas)', conn)['name'].tolist()

print(f' Banco criado: transporte_v2.db')
print(f'   Total de registros : {total:,}')
print(f'   Total de colunas   : {len(colunas)}')
print(f'\nColunas gravadas:')
print(colunas)
conn.close()

In [ ]:
# 5.4 — Verificação final das flags binárias
conn = sqlite3.connect('transporte_v2.db')
checks = pd.read_sql("""
    SELECT
        COUNT(*)                   AS total_registros,
        SUM(is_completed)          AS corridas_concluidas,
        SUM(cancelled_by_driver)   AS canceladas_motorista,
        SUM(cancelled_by_customer) AS canceladas_cliente,
        SUM(no_driver_found)       AS sem_motorista
    FROM fato_corridas
""", conn)
conn.close()

print(' Verificação final do banco transporte_v2.db:')
print(checks.T.to_string(header=False))
print('\n🎉 ETL concluído!')

## 6. Resumo dos dados

| Métrica | Valor |
|---|---|
| Total de Corridas | 150.000 |
| Taxa de Conclusão | 62,0% |
| Taxa Cancelamento (Motorista) | 18,0% |
| Taxa Cancelamento (Cliente) | 7,0% |
| Taxa "No Driver Found" | 7,0% |
| Receita Total (concluídas) | ~R$ 51,8M |
| Hora de Pico Crítica | 18h–19h |
| Veículo com Maior Receita | Auto |

### 🔑 Insights Chave:
1. **38% das corridas não são concluídas** — impacto direto na receita
2. **VTAT alto → mais cancelamentos**: tempo de chegada >10min eleva fortemente a taxa de não conclusão
3. **18h–20h** é o horário mais crítico para ausência de motoristas
4. **"Customer related issue"** lidera os cancelamentos por motorista — sinal de matching inadequado